# RAG Chatbot with Mistral-7B and FAISS



## Objective

This notebook builds a question-answering chatbot using a Retrieval-Augmented Generation (RAG) pipeline over the [Amazon Q&A dataset](https://huggingface.co/datasets/sentence-transformers/amazon-qa). All models are free and open-source.

The pipeline works as follows:
1. Encode a knowledge base of Amazon Q&A pairs into dense vector embeddings
2. At query time, retrieve the most semantically similar Q&A pairs using FAISS
3. Feed the retrieved context and the user question to Mistral-7B-Instruct to generate a grounded answer

We also compare two chatbot variants:
- **Stateless** : each question is answered independently, with no memory of previous turns
- **Memory-augmented** : the last 3 conversation turns are appended to the prompt to maintain context across exchanges

---

## Architecture Overview

```
User Query
    ↓
Sentence Embedding (all-MiniLM-L6-v2)
    ↓
FAISS Similarity Search (Top-k Q&A pairs from Amazon dataset)
    ↓
Prompt Construction (context + optional history + query)
    ↓
Mistral-7B-Instruct → Final Answer
```



In [1]:
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [2]:
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.7 MB/s eta 0:00:00


## 1. Import Libraries

In [3]:
from datasets import load_dataset
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

## 2. Load the Dataset

We use the [Amazon Q&A dataset](https://huggingface.co/datasets/sentence-transformers/amazon-qa) from Hugging Face. It contains real user questions and answers about Amazon products, making it a suitable knowledge base for a product-oriented chatbot.

We load **20,000 examples** using streaming mode to avoid downloading the full dataset.

In [4]:
DATA_SIZE = 20000

dataset = load_dataset(
    "sentence-transformers/amazon-qa",
    split="train",
    streaming=True
)

data = []
for i, item in enumerate(dataset):
    if i >= DATA_SIZE:
        break
    data.append(f"Q: {item['query']} A: {item['answer']}")

print(f"Loaded {len(data)} Q&A pairs.")
print("Sample:", data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Loaded 20000 Q&A pairs.
Sample: Q: does this fit the z2x version? Thx A: I am not 100% sure. It appears that it does based on the size of the torch housing. I have the 9p and it fits perfectly. Compare the two and see. Otherwise it is a great product and has lasted 2 years so far wearing every day on my duty belt. Gets lots of inquiries and compliments.


## 3. Generate Text Embeddings

We use `all-MiniLM-L6-v2` to encode each Q&A pair into a dense vector. At retrieval time, the user query is embedded with the same model, and cosine similarity is used to find the closest entries in the knowledge base.

In [5]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedder.encode(data, show_progress_bar=True)

print(f"Embedding shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Embedding shape: (20000, 384)


## 4. Build the FAISS Index

We index the embeddings using FAISS, which enables fast nearest-neighbor search over large vector collections. `IndexFlatL2` performs exact search using L2 distance.

In [6]:
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

print(f"Index built with {index.ntotal} vectors.")

Index built with 20000 vectors.


### Retrieval Function

Given a user query, this function encodes it using the same embedding model and retrieves the top-k most similar Q&A pairs from the FAISS index.

In [7]:
def retrieve(query, k=3):
    query_embedding = embedder.encode([query])
    D, I = index.search(query_embedding, k)
    return [data[i] for i in I[0]]

## 5. Load the Language Model

We use `Mistral-7B-Instruct-v0.1`, a 7-billion parameter instruction-tuned model. It is freely available on Hugging Face with no access restrictions.

> **Note:** The model weights are approximately 14GB. Downloading may take several minutes.

In [8]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

## 6. Answer Generation

We define two variants of the generation pipeline.

---

### 6.1 Stateless Chatbot

Each question is answered independently. The prompt contains only the retrieved context and the current query, there is **no memory of previous exchanges**. Vague follow-up questions are re-embedded and retrieved independently, which can lead to topic drift.

In [9]:
def generate_answer(query):
    #answer a query using retrieved context only
    context = "\n".join(retrieve(query))

    prompt = f"""
    [INST]
    You are a helpful assistant.

    Rules:
    - Use the context if it helps.
    - If the context is not enough, use your knowledge.
    - Be concise and friendly.
    - Answer in ONE short sentence.

    Context:
    {context}

    Question:
    {query}
    [/INST]
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return result.split("[/INST]")[-1].strip()

### 6.2 Memory-Augmented Chatbot

This variant maintains a rolling history of the last 3 conversation turns. Each prompt includes both the retrieved context **and** the recent dialogue, allowing the model to resolve references and remain consistent across follow-up questions.

In [10]:
conversation_history = []

def generate_answer_with_memory(query):
    global conversation_history

    if conversation_history:
        last_q, last_a = conversation_history[-1]
        retrieval_query = f"{last_q} {last_a} {query}"
    else:
        retrieval_query = query

    context = "\n".join(retrieve(retrieval_query))

    history = ""
    for q, a in conversation_history[-3:]:
        history += f"User: {q}\nAssistant: {a}\n"

    prompt = f"""
    [INST]
    You are a helpful assistant.

    Rules:
    - Use the context if it helps.
    - If the context is not enough, use your knowledge.
    - Be concise and friendly.
    - Answer in ONE short sentence.

    Conversation history:
    {history}

    Context from knowledge base:
    {context}

    Question:
    {query}
    [/INST]
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = result.split("[/INST]")[-1].strip()

    conversation_history.append((query, answer))

    return answer

## 7. Evaluation

We run both variants through the same 3-question conversation. The questions are designed to test context retention: starting with a broad recommendation, then asking follow-ups that only make sense in relation to the first answer.

---
### 7.1 Stateless Results

Without access to prior turns, Q2 and Q3 are treated as standalone queries. The retriever independently matches them against the knowledge base, in this case pulling laptop-related content, causing the conversation to drift away from the phone mentioned in Q1.

In [11]:
print("Q1 :", generate_answer("what phone do you recommend?"))
print("Q2 :", generate_answer("what about battery life?"))
print("Q3 :", generate_answer("is it good for gaming?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q1 : I recommend the unlocked/unbranded Galaxy NOTE II for international travel as it accepts any SIM (micro) and has no contract to sign.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q2 : Answer:
    Battery life can vary depending on usage and battery health. It's recommended to use the laptop connected to an AC outlet when possible to prolong battery life.
Q3 : Based on the context, it seems that the laptop is capable of handling popular games like Minecraft and even complex online games like Half-Life and Halo. However, it may not be as good for more demanding games like Sims


### 7.2 Memory-Augmented Results

With conversation history included in the prompt, the model can resolve the ambiguous follow-ups ("what about battery life?", "is it good for gaming?") in relation to the phone recommended in Q1, producing consistent and contextually appropriate answers.

In [12]:
# reset history
conversation_history = []

print("Q1 :", generate_answer_with_memory("what phone do you recommend?"))
print("Q2 :", generate_answer_with_memory("what about battery life?"))
print("Q3 :", generate_answer_with_memory("is it good for gaming?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q1 : I recommend the unlocked/unbranded Galaxy NOTE II for international travel as it accepts any SIM (micro) and has no contract to sign.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q2 : The unlocked/unbranded Galaxy NOTE II is known for its long battery life.
Q3 : Based on the context from the knowledge base, the unlocked/unbranded Galaxy NOTE II is not recommended for gaming as it is not a gaming phone.


## 8. Conclusion

### What we built

A RAG chatbot that:
- Encodes a 20K-entry knowledge base using `all-MiniLM-L6-v2`
- Retrieves relevant context via fast nearest-neighbor search with FAISS
- Generates answers with `Mistral-7B-Instruct`, conditioned on the retrieved context

### Key takeaway: conversational context matters

The comparison between the two variants shows the direct impact of including conversation history in the prompt:

| | Stateless | Memory-augmented |
|---|---|---|
| Q1: phone recommendation | Answers correctly | Answers correctly |
| Q2: battery life? | Drifts to laptops (context lost) | Stays on the recommended phone |
| Q3: good for gaming? | Still about laptops | Consistent with previous turns |

The stateless version treats each question in isolation. Vague follow-ups like "what about battery life?" are re-embedded and matched independently by the retriever, in this case surfacing laptop-related content. The memory-augmented version injects prior turns into the prompt, anchoring the model to the phone discussed in Q1.

### Limitations & next steps

- **Retrieval quality**: Short or ambiguous queries can surface irrelevant results. A reranker or hybrid BM25+dense retrieval strategy would improve precision.
- **Memory depth**: A 3-turn window is sufficient for short conversations. For longer sessions, a summarization-based memory approach would be more scalable.
- **Model**: Mistral-7B-Instruct is a strong open-source baseline, but a larger or domain-fine-tuned model would likely produce more accurate and consistent answers.